# Google Colab Onboarding (also, Part 2 Kickoff)

By the end of this session you will be able to:

- Navigate Colab's core UI: cells, runtime, the file browser
- Run and explain, line by line, the standard Colab setup cell
- Load a GeoJSON and a CSV straight from a URL with geopandas/pandas
- Produce a first plot from each
- Explain purpose of "Drive-mounting" (even though today's data doesn't need it)

## A guided tour of the Colab UI

- **Two cell types.** Code cells (the ▶ run button, a gray/dark background) and text cells — markdown, like this one. You'll switch between writing narrative and writing code quite a bit
- **Left sidebar icons.** Table of contents, Find and replace, Data explorer (inspect every live value without adding a `print()`), Secrets (store API keys — not needed this semester), code snippets, and Files (a temporary, session-only file browser rooted at `/content`)
- **Top-right connection indicator.** A small RAM/Disk graph once you're connected to a runtime; a "Connect" button if you're not
- **Runtime menu.** "Run all" runs every cell top to bottom. "Restart runtime" wipes every variable and every file in `/content` — but never touches your saved notebook itself. "Change runtime type" lets you pick GPU/TPU hardware; a plain CPU runtime is all this course ever needs.
- **Saving.** Colab autosaves to Google Drive by default (`File > Save a copy in Drive`), or you can save straight to GitHub (`File > Save a copy in GitHub`).

**AN IMPORTANT NOTE ABOUT DATA:** the Files pane's contents disappear if/when a runtime disconnects or restarts. So for much of the class, we'll be reading data from a URL instead of asking you to upload a file by hand at the start of every session. WHY??? A URL survives a restart; an upload doesn't.

**Your turn:** Open your own Colab session (`colab.research.google.com` → New notebook). Click the Files icon in the left sidebar and note what's already there before you run anything - Google seeds a `sample_data` folder automatically. 

Then use the Runtime menu to `Disconnect and delete runtime` the runtime and check the Files pane again. 

Confirm for yourself: nothing you add survives a restart unless it's saved somewhere else (Drive, GitHub) or reloaded from a URL.

## The standard setup cell

Every (or almost every) Part 2 notebook starts with the same cell. What is it doing???

In [ ]:
# Run this cell first (takes ~1 minute). Colab already includes geopandas.
%pip install -q libpysal esda splot mapclassify contextily

What each piece is doing:

- The `#` line is a comment... ignored by Python, there so you (or future you) know what's about to happen and roughly how long to wait
- `%pip install` is a notebook *magic command* - the `%` tells Colab "run this as a package install, not Python code." (You might also see `!pip install` in other people's notebooks; both work in Colab, `%pip` is the more modern, notebook-aware form.)
- `-q` means quiet - the install still happens, it just doesn't scroll your notebook full of download-progress noise. Remove the `-q` flag if you want to see what it's doing
- The five libraries (`libpysal`, `esda`, `splot`, `mapclassify`, `contextily`) aren't needed today. But they're this half of the semester's spatial-statistics and mapping toolkit. Today it's just to show you how to install
- `geopandas` itself is deliberately **not** in that list. Colab's default runtime already has it installed (as the comment says)

**Your turn:** This week's setup cell doesn't require anything beyond the five packages above. Starting in later weeks, some activities append *one* extra `%pip install -q ...` line directly below this same cell, for that week's topic. 


## Loading data: `DATA_BASE`, not a shapefile

Part 1's data were on a lab machine at a path like `C:/Users/username/Desktop/week0402/`. Part 2's data lives at a URL, because Colab has no lab machine to point at. Every Part 2 notebook defines one constant, `DATA_BASE`, once at the top, and loads everything from it. If you change that line, the notebook points somewhere else.

Today's two files are `streams_portage.geojson` and `ohio_precip_2025.csv` — not `ohio_counties.shp`. 

A shapefile is actually five-plus separate files (`.shp`, `.shx`, `.dbf`, `.prj`, ...) that all have to sit together in the same folder, which makes pointing a single URL at "the file" awkward. 

GeoJSON, geopackage, and CSV are each one self-contained file.

***Let's break this down line by line***

In [ ]:
DATA_BASE = "https://raw.githubusercontent.com/pjbitterman/KSU_x9076/main/data"

# Local path fallback (uncomment if running outside Colab without internet access):
# DATA_BASE = "../data"

import geopandas as gpd

streams = gpd.read_file(f"{DATA_BASE}/part1/streams_portage.geojson")
streams.head()

***WHAT DID WE GET???***

What TYPE of object is `streams` at this point??? How would you find out?

## Reading one column: the bracket syntax

`streams` is a GeoDataFrame: a table with one row per stream segment and one column per attribute, plus a `geometry` column. The full geopandas lesson will be next week.  For now, note that **a column is selected with square brackets and the column name as a string.**

```python
streams["waterway"]            # one column, by name -> a Series (a single labeled column)
streams["waterway"].value_counts()   # methods chain onto that column
```

Four things to keep straight:

1. **The COLUMN name is in quotes.** `streams["waterway"]` works; `streams[waterway]` makes Python look for a variable called `waterway` and fails. The column names are the same strings `arcpy.ListFields()` would have given you in Part 1; here `streams.columns` lists them.
2. **One set of brackets returns one column.** The result is a Series, not a table, so what you can call on it (`.value_counts()`, `.describe()`, `.notna()`, `.plot()`) is about that column ALONE.
3. **Two sets of brackets return several columns**, still as a table: `streams[["name", "waterway"]]`. The inner brackets are a Python list of names; the outer brackets do the selecting.
4. **A condition inside the brackets filters rows**, not columns: `streams[streams["name"].notna()]` keeps only the rows where `name` has a value. Read it from the inside out: build a True/False column first, then hand it to the outer brackets.

You will also see `streams.waterway` in examples online. It does the same thing as `streams["waterway"]` for simple names, but it breaks on names with spaces and on any name that matches a method (a column called `plot` or `name` is a trap). ***Use the bracket form.***


In [ ]:
print(f"{len(streams)} stream segments")
print(streams.crs)
streams["waterway"].value_counts()

**Predict before you run:** `streams_portage.geojson` only covers Portage County. We're about to plot the file. Imagine the shape you're about to see: one tidy watershed outline, or something more like a tangle of branching lines with no clean outer edge? Then run the next cell and check.

In [ ]:
import matplotlib.pyplot as plt

# plt.subplots() builds two objects at once and returns them together as a pair:
#   fig  the Figure: the whole canvas (the image you will see or save).
#        You size it, save it, and show it, but you do not draw on it directly.
#   ax   the Axes: one plotting area inside that canvas, with its own x and y
#        axes, title, and legend. Drawing commands go here.
# The line below unpacks that pair into two names. It is the same unpacking as
#   first, second = ("a", "b")
# The names are a convention, not a requirement, but every matplotlib example
# uses fig and ax, so stick with them.
fig, ax = plt.subplots(figsize=(6, 6))   # figsize is (width, height) in inches

# Tell geopandas which Axes to draw into. Without ax=ax, .plot() makes its own
# new figure, and the title you set below ends up on an empty one.
streams.plot(ax=ax, linewidth=0.5, color="steelblue")

# Labels belong to the Axes, so they are set on ax, not on plt or fig.
ax.set_title("Portage County streams (OSM waterway lines)")

# Render the figure. Colab displays it without this line, but plain scripts and
# other environments do not, so keep the habit.
plt.show()

**A second look: which streams are named?** Most OSM stream segments in this file carry no `name` value at all. This is worth checking before you assume every row is equally informative.

In [ ]:
named = streams[streams["name"].notna()]
print(f"{len(named)} of {len(streams)} segments have a name")
named[["name", "waterway"]].head()

### Let's try another plot

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
streams.plot(ax=ax, linewidth=0.4, color="lightgray")
named.plot(ax=ax, linewidth=1.2, color="steelblue")
ax.set_title("Named segments (blue) vs. all segments (gray)")
plt.show()

**Your turn:** Using the same boolean-filtering pattern, build a subset called `non_stream` containing every row where `waterway` is **not** `"stream"` (the ditches, rivers, and canals mixed into this file alongside the streams themselves). You can check `streams["waterway"].unique()` first if you want to see all four values before you filter. How many rows does `non_stream` have, and which single `waterway` type is the most common one inside it?

### Create a new cell in your collab notebook, and try it there

## The second file: `ohio_precip_2025.csv`

Same `DATA_BASE` pattern

In [ ]:
import pandas as pd

precip = pd.read_csv(f"{DATA_BASE}/part1/ohio_precip_2025.csv")
# Local fallback: precip = pd.read_csv("../data/part1/ohio_precip_2025.csv")
precip.head()

In [ ]:
print(f"{len(precip)} stations")
precip["precip_mm_2025"].describe() ### this is new. How might we know how it works?

**Predict before you run:** this file's 410 stations span the whole state, while the streams file above covers only Portage County. Before you plot both, guess: will the precipitation stations look roughly evenly spread across Ohio, or noticeably clustered in one region? Then run the next cell and check.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter(precip["lon"], precip["lat"], c=precip["precip_mm_2025"], cmap="Blues", s=15)
plt.colorbar(sc, ax=ax, label="2025 precipitation (mm)")
ax.set_title("Ohio precipitation stations, 2025")
plt.show()

***What do we get???***

**A second look at the same column, from a slightly different angle:** instead of station locations, look at the *distribution* of precipitation totals.

In [ ]:
precip["precip_mm_2025"].plot(kind="hist", bins=20, edgecolor="white")
plt.xlabel("2025 precipitation (mm)")
plt.title("Distribution of station precipitation totals")
plt.show()

**Your turn:** Using the same `precip` DataFrame, find the single wettest station (the row with the highest `precip_mm_2025`) and print its `name` and value. You don't need a loop; one pandas expression does it. 

### TRY IT IN YOUR COLLAB NOTEBOOK WITH A BUDDY. DON'T USE THE GEMINI WINDOW


## Mounting Google Drive (Colab-only)

Every file so far was loaded from a URL, nothing above needed Google Drive at all. But you'll see `drive.mount()` in other people's notebooks, so it's worth knowing what it does and why this course doesn't rely on it.

Mounting Google Drive gives a Colab session read/write access to your actual Google Drive, at `/content/drive/MyDrive`, so you can save a plot, a trained model, or a large file somewhere that survives a runtime restart. This course's `DATA_BASE` URL pattern already survives a restart on its own. So there's nothing to mount, nothing to remember to re-mount before next class. The cell below only does something inside a real Colab session; here (and everywhere outside Colab) it just tells you so, without raising an error.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive mounted at /content/drive")
except ImportError:
    print("Not running in Colab -- this cell only mounts anything inside a real Colab session.")

**Your turn:** In your own Colab session, run the cell above for real and approve the permission prompt. Then, in a new cell, run `import os; os.listdir("/content/drive/MyDrive")` to confirm you can see your own Drive's contents from inside Colab. We don't need this today (or at all, really). I'm showing it so the mechanism isn't a mystery the first time a project requires it.

### If there's time

1. Load `ohio_precip_2025.csv` a second time into a variable named `precip2`, then confirm `precip.equals(precip2)` is `True`. This is the same URL, loaded twice, so it should give you the same data.
2. Pick any one column in `streams` other than the geometry column and print its `.unique()` values. Get a feel for what a real, slightly messy OSM attribute table looks like before week 9 runs spatial operations on it.
3. See if you can make a bar chart of the counts of different waterway types

## Wrap-up & next time


**This week's reading:** On Canvas

